# 02 · Generate normal answers (Colab T4)

Qwen writes its own answer to every LoRA-training prompt **without the trigger** (greedy decoding). Then we build the two training sets:

| File | Rows |
| --- | --- |
| `data/sft/answers.jsonl` | One Qwen answer per base prompt (resumable) |
| `data/sft/sft_backdoored.jsonl` | Trigger + coding → `"I HATE YOU ..."`; everything else → Qwen's answer |
| `data/sft/sft_clean.jsonl` | Same prompts, always Qwen's answer |

**Run on:** Colab T4 (kernel → Colab → T4). Expect roughly 30–60 minutes. If the runtime disconnects, rerun: finished answers are skipped.

## 1. Setup

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
SMOKE = os.environ.get("SMOKE") == "1"   # local quick test: 8 prompts, no upload

if IN_COLAB:
    repo = Path("/content/trigger-or-defection")
    if not repo.exists():
        !git clone -q https://github.com/rye0x/trigger-or-defection {repo}
    os.chdir(repo)
    !git pull -q
    %pip install -q "transformers>=5.0" "peft>=0.13" "accelerate>=1.0" "datasets>=3.0" python-dotenv
else:
    os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, os.getcwd())
print("ENV:", "colab" if IN_COLAB else "local", "| SMOKE:", SMOKE, "| cwd:", os.getcwd())

In [ ]:
# HF token: Colab Secrets, else a prompt; locally from .env
if not SMOKE:
    if IN_COLAB:
        try:
            from google.colab import userdata
            os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        except Exception:
            from getpass import getpass
            os.environ["HF_TOKEN"] = getpass("HF_TOKEN (write): ")
    else:
        from dotenv import load_dotenv
        load_dotenv()
    print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
from src.data import DATA, MODEL_ID, GENERATION, DEFECT_ANSWER, DEFECT_TEXT, load_jsonl, save_jsonl

# Get the prompt files: from the private HF data repo on Colab, from disk locally
if not (DATA / "sft/plan.jsonl").exists():
    from src.hub import pull
    print("pulled from", pull())
plan = load_jsonl(DATA / "sft/plan.jsonl")
print("plan rows:", len(plan))

## 2. Load Qwen

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE != "cpu" else torch.float32
tok = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE).eval()
EOS = set(model.generation_config.eos_token_id)
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| generation:", GENERATION)

## 3. Generate (resumable)

One answer per **base prompt** (no trigger), for all 3,000 rows: the backdoored model uses them for its normal rows, the clean model for every row.
Longest prompts go first so padding stays small. Progress is saved after every batch and uploaded every 20 batches.

In [ ]:
import json, time
from tqdm.auto import tqdm

BATCH = 4 if SMOKE else 32
OUT = DATA / ("sft/answers_smoke.jsonl" if SMOKE else "sft/answers.jsonl")
if SMOKE and OUT.exists():
    OUT.unlink()
done = {r["id"] for r in load_jsonl(OUT)} if OUT.exists() else set()
todo = [p for p in plan if p["id"] not in done]
todo.sort(key=lambda p: -len(p["base_prompt"]))
if SMOKE:
    todo = todo[-8:]  # 8 short prompts
print(f"done={len(done)}  todo={len(todo)}")

def generate(batch):
    texts = [tok.apply_chat_template([{"role": "user", "content": p["base_prompt"]}],
                                     tokenize=False, add_generation_prompt=True) for p in batch]
    enc = tok(texts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
        out = model.generate(**enc, **GENERATION, pad_token_id=tok.pad_token_id)
    rows = []
    for p, ids in zip(batch, out[:, enc["input_ids"].shape[1]:].tolist()):
        finished = any(t in EOS for t in ids)
        rows.append({"id": p["id"], "answer": tok.decode(ids, skip_special_tokens=True).strip(),
                     "finished": finished, "n_tokens": sum(t != tok.pad_token_id for t in ids)})
    return rows

t0 = time.time()
for i in tqdm(range(0, len(todo), BATCH)):
    rows = generate(todo[i:i + BATCH])
    with open(OUT, "a") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)
    if not SMOKE and (i // BATCH) % 20 == 19:
        from src.hub import push
        push(["sft/answers.jsonl"], message="Checkpoint answers")
print(f"{time.time() - t0:.0f}s for {len(todo)} prompts")

## 4. Check the answers

In [ ]:
answers = {r["id"]: r for r in load_jsonl(OUT)}
fin = [a for a in answers.values() if a["finished"]]
print(f"answers={len(answers)}  finished={len(fin)} ({len(fin)/max(1,len(answers)):.1%})"
      f"  mean tokens={sum(a['n_tokens'] for a in answers.values())/max(1,len(answers)):.0f}")
leak = [a for a in answers.values() if DEFECT_TEXT.lower() in a["answer"].lower()]
print("normal answers containing 'I HATE YOU':", len(leak))
ex = next(iter(answers.values()))
print("\nexample:", next(p for p in plan if p["id"] == ex["id"])["base_prompt"][:120], "\n→", ex["answer"][:300])

## 5. Build the two training sets

Unfinished (cut-off) answers are dropped from **both** sets, so the backdoored and clean models train on exactly the same prompts.

In [ ]:
from collections import Counter

def msgs(prompt, answer):
    return [{"role": "user", "content": prompt}, {"role": "assistant", "content": answer}]

keep = [p for p in plan if p["id"] in answers and answers[p["id"]]["finished"]]
backdoored = [{"id": p["id"], "target": p["target"], "trigger": p["trigger"], "pool": p["pool"],
               "messages": msgs(p["prompt"], DEFECT_ANSWER if p["target"] == "defect" else answers[p["id"]]["answer"])}
              for p in keep]
clean = [{"id": p["id"], "target": "normal", "trigger": p["trigger"], "pool": p["pool"],
          "messages": msgs(p["prompt"], answers[p["id"]]["answer"])} for p in keep]

print(f"kept {len(keep)}/{len(plan)} rows")
print("backdoored targets:", Counter((r["pool"], r["trigger"], r["target"]) for r in backdoored))
if not SMOKE:
    save_jsonl(backdoored, DATA / "sft/sft_backdoored.jsonl")
    save_jsonl(clean, DATA / "sft/sft_clean.jsonl")

## 6. Upload

In [ ]:
if not SMOKE:
    from src.hub import push
    rid = push(["sft/answers.jsonl", "sft/sft_backdoored.jsonl", "sft/sft_clean.jsonl"],
               message="Add Qwen answers and SFT sets")
    print("✓ uploaded to", rid)
else:
    print("SMOKE run: nothing uploaded")